# Assignment 4 — Evaluation

Seven questions covering Dostoevsky and Nietzsche. For each we record:

1. The question.
2. The retrieved chunks (top-k with similarity scores).
3. The final generated answer.
4. A manual accuracy judgement: **Correct / Partial / Incorrect**.

**Prerequisite:** the vLLM server must be running locally (defaults to `http://localhost:8000/v1`). See the README for the launch command.

In [ ]:
import sys, os, pathlib

# Ensure the project root is on sys.path so `import src.rag` works.
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.rag import RAG
rag = RAG()
print("RAG ready. Index size:", rag.index.ntotal, "chunks")

In [ ]:
import textwrap
import pandas as pd
from IPython.display import Markdown, display

EVAL_QUESTIONS = [
    "What does Raskolnikov mean when he talks about 'extraordinary men'?",
    "How does Ivan Karamazov justify his rebellion against God?",
    "What is the underground man's complaint about the 'man of action'?",
    "What does Nietzsche mean by the 'eternal recurrence' in Thus Spake Zarathustra?",
    "In Beyond Good and Evil, what does Nietzsche say about master and slave morality?",
    "In The Antichrist, what is Nietzsche's main critique of pity?",
    "How does Sonia respond when Raskolnikov confesses the murder to her?",
]

def show(result):
    display(Markdown(f"### Q: {result['question']}"))
    display(Markdown("**Retrieved passages**"))
    for c in result['chunks']:
        snippet = textwrap.shorten(c['text'].replace('\n', ' '), width=240)
        display(Markdown(
            f"- [{c['rank']}] *{c['author']} — {c['title']}* "
            f"(chunk {c['chunk_index']}, score {c['score']:.3f}) — {snippet}"
        ))
    display(Markdown("**Answer**"))
    display(Markdown(result['answer']))

In [ ]:
results = []
for q in EVAL_QUESTIONS:
    r = rag.answer(q)
    show(r)
    results.append(r)

## Accuracy summary

Fill in the `verdict` column manually after reading each answer above. The summary table is what goes into the report.

In [ ]:
summary = pd.DataFrame({
    "question": [r['question'] for r in results],
    "top_chunk_source": [
        f"{r['chunks'][0]['author']} — {r['chunks'][0]['title']}" if r['chunks'] else "(none)"
        for r in results
    ],
    "top_score": [r['chunks'][0]['score'] if r['chunks'] else 0.0 for r in results],
    "verdict": ["" for _ in results],   # fill: Correct / Partial / Incorrect
})
summary

In [ ]:
summary.to_csv('eval_results.csv', index=False)
print('Wrote eval_results.csv')